In [142]:
import os
import sys
# install packages
%pip install psycopg2 networkx matplotlib numpy openpyxl
import psycopg2
import math
from datetime import datetime

import numpy as np
import pandas as pd
import pylab as pl
import matplotlib.pyplot as plt
from matplotlib import collections as mc
import matplotlib.ticker as plticker
from matplotlib.lines import Line2D

# packages needed for utils
%pip install nltk pandas scikit-learn textblob
sys.path.insert(0, os.path.abspath(
    '.'))  # add the current module so that we can import the utils file
from utils import get_top_keywords_for_query, create_df_for_query
from db_setup import get_connection


Note: you may need to restart the kernel to use updated packages.



In [2]:
conn = get_connection()
start_year = 2014  # inclusive
end_year = 2025  # inclusive

In [ ]:
venues = ['CCGrid', 'ATC', 'DSN', 'Euro-Par', 'EuroSys', 'FAST', 'FGCS', 'HiPC', 'HPDC', 'IC2E',
    'ICDCS', 'ICPP', 'INFOCOM', 'IPDPS', 'NDSS', 'OSDI', 'POPL', 'PPOPP', 'SC', 'SIGMETRICS', 'SoCC',
    'SOSP', 'SYSTOR', 'TPDS', 'TJS', 'JPDC']

venue_count_query = """
SELECT venue, COUNT(*) AS count_per_venue FROM publications 
WHERE venue IN %s
GROUP BY venue 
ORDER BY count_per_venue 
DESC;
"""
# Get amount of papers for each venue in total
with conn:
    with conn.cursor() as cursor:
        cursor.execute(venue_count_query, (tuple(venues),))
        venues_rows = cursor.fetchall()
        print(venues_rows)

[('INFOCOM', 9901), ('FGCS', 7497), ('TPDS', 5587), ('IPDPS', 5519), ('TJS', 4860), ('JPDC', 4739), ('SC', 3931), ('ICDCS', 3692), ('Euro-Par', 2817), ('CCGrid', 2594), ('ICPP', 2520), ('SoCC', 2128), ('SIGMETRICS', 1977), ('POPL', 1963), ('DSN', 1767), ('HPDC', 1544), ('HiPC', 1462), ('EuroSys', 1351), ('SOSP', 1008), ('PPOPP', 595), ('IC2E', 527), ('SYSTOR', 360), ('NDSS', 224), ('OSDI', 39), ('FAST', 11), ('ATC', 7)]


In [133]:


query1 = ("""
SELECT * FROM publications
    WHERE venue IN %s 
    AND year >= %s 
    AND (lower(title) ILIKE %s OR lower(abstract) ILIKE %s)

    AND ((lower(title) ILIKE %s OR lower(abstract) ILIKE %s)
    OR (lower(title) ILIKE %s OR lower(abstract) ILIKE %s)
    OR (lower(title) ILIKE %s OR lower(abstract) ILIKE %s)
    OR (lower(title) ILIKE %s OR lower(abstract) ILIKE %s))

    AND ((lower(title) ILIKE %s OR lower(abstract) ILIKE %s)
    OR (lower(title) ILIKE %s OR lower(abstract) ILIKE %s)
    OR (lower(title) ILIKE %s OR lower(abstract) ILIKE %s)
    OR (lower(title) ILIKE %s OR lower(abstract) ILIKE %s)
    OR (lower(title) ILIKE %s OR lower(abstract) ILIKE %s))
    
    AND NOT ((lower(title) ILIKE %s OR lower(abstract) ILIKE %s)
    OR (lower(title) ILIKE %s OR lower(abstract) ILIKE %s)
    OR (lower(title) ILIKE %s OR lower(abstract) ILIKE %s)
    OR (lower(title) ILIKE %s OR lower(abstract) ILIKE %s)
    OR (lower(title) ILIKE %s OR lower(abstract) ILIKE %s)
    OR (lower(title) ILIKE %s OR lower(abstract) ILIKE %s)
    OR (lower(title) ILIKE %s OR lower(abstract) ILIKE %s)
    OR (lower(title) ILIKE %s OR lower(abstract) ILIKE %s))
    
 ;
""",  
[tuple(venues), 
start_year,
'%schedul%', '%schedul%',

'%workflow%', '%workflow%',
'%batch%', '%batch%',
'%spark%', '%spark%',
'%backfilling%', '%backfilling%',

'%fair%', '%fair%',
'%slowdown%', '%slowdown%',
'%response%', '%response%',
'%throughput%', '%throughput%',
'%shared%', '%shared%',

'%deep%', '%deep%',
'%dnn%', '%dnn%',
'%gpu%', '%gpu%',
'%cuda%', '%cuda%',
'%bandwidth%', '%bandwidth%',
'% edge%', '% edge%',
'% fog%', '% fog%',
'% tcp%', '% tcp%',

])

find_paper_doi_query = """
SELECT * FROM publications WHERE doi = %s;
"""


# Missing:     "10.1109/TSC.2018.2866421"
quasi_gold_standard_doi = [
    "10.1016/j.future.2015.08.009",
    "10.1016/j.jpdc.2024.104968",
    "10.1007/s11227-018-2604-2",
    "10.1109/CCGrid49817.2020.00-36",
    "10.1109/CCGrid.2015.123",


    "10.1109/CCGRID.2018.00048",



    "10.1109/TPDS.2013.88",

    "10.1109/TPDS.2020.3041829",
    "10.1109/TPDS.2018.2846234",
    "10.1109/CCGrid.2015.143",
    "10.1109/CCGRID.2019.00010",
    "10.1109/CCGrid49817.2020.00-23",
    "10.1109/CCGrid.2015.27",
]

#  Abstract or keywords truncated
quasi_gold_truncated_doi = [
    "10.1109/TPDS.2018.2846234",
    "10.1109/CCGrid.2015.143",
    "10.1109/CCGRID.2019.00010",
    "10.1109/CCGrid49817.2020.00-23",
    "10.1109/CCGrid.2015.27",
    "10.1109/TPDS.2020.3041829",
]



In [97]:
# get the quasi gold standard
quasi_gold_standard = []
for standard in quasi_gold_standard_doi:
    with conn.cursor() as cursor:
        
        cursor.execute(find_paper_doi_query, (standard,))
        full_paper = cursor.fetchall()
        if len(full_paper) == 1:
            quasi_gold_standard.append(full_paper[0])
        else:
            print(f"Missing entry for:{standard}")


assert(len(quasi_gold_standard_doi) == len(quasi_gold_standard))

In [ ]:
# Construct the corpus dataframe that we want to run TF-IDF against
corpus_query = """
    SELECT * 
    FROM publications 
    WHERE year BETWEEN %s AND %s"""
corpus_df = create_df_for_query(conn, corpus_query, [start_year, end_year])



c:\Users\dkazemak\Projects\aip_v2\AIP\notebooks\utils.py:92: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, database_connection, params=query_params)


In [29]:
# see most common keywords for these papers
custom_stop_words = ["resource", "scheduling", "scheduler", "job", "performance", "algorithm", "time", "system", "application", "task"]

keywords = get_top_keywords_for_query(conn, corpus_df, query1[0], 50, query1[1])
filtered_keywords = [keyword for keyword in keywords if keyword not in custom_stop_words]
print(filtered_keywords)

c:\Users\dkazemak\Projects\aip_v2\AIP\notebooks\utils.py:92: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, database_connection, params=query_params)


['workflow', 'makespan', 'cloud', 'batch', 'computing', 'execution', 'schedule', 'data', 'objective', 'user', 'based', 'multi', 'optimization', 'problem', 'environment', 'cost', 'solution', 'service', 'proposed', 'fairness', 'throughput', 'scientific', 'workload', 'model', 'different', 'utilization', 'completion', 'heuristic', 'heterogeneous', 'minimize', 'cluster', 'parallel', 'efficient', 'energy', 'response', 'policy', 'multiple', 'strategy', 'aware', 'machine']


In [134]:
# Find all papers that fit in the query
retrieved_papers = []
with conn.cursor() as cursor:
    # conn.rollback()
    cursor.execute(query1[0], (*query1[1], ))
    retrieved_papers.extend(cursor.fetchall())
print(len(retrieved_papers))


67


In [137]:
# check which papers are missing
relevant_retrieved_doi = [paper[5] for paper in retrieved_papers if paper[5] in quasi_gold_standard_doi]
for paper in quasi_gold_standard:
    if paper[5] not in relevant_retrieved_doi and paper[5] not in quasi_gold_truncated_doi:
        print(paper)
        
# print found papers
for i, paper in enumerate(retrieved_papers):
    if i > 30 or i < 19:
        continue
    print(paper[4])


#calculate sensitivity and precision
number_retrieved = len(retrieved_papers) + len(quasi_gold_truncated_doi) # add truncated manually, to still have correct calculations
number_relevant = len(quasi_gold_standard)
number_relevant_retrieved = len(relevant_retrieved_doi) + len(quasi_gold_truncated_doi) # add truncated manually, to still have correct calculations
print(f"Number retrieved: {number_retrieved}\nNumber of relevant: {number_relevant}\nNumber of relevant retrieved: {number_relevant_retrieved}")

quasi_sensitivity = number_relevant_retrieved / number_relevant
quasi_precision = number_relevant_retrieved / number_retrieved
print(f"sensitivty: {round(quasi_sensitivity,3) * 100}%\nprecision: {round(quasi_precision, 3)* 100}%")


Helping HPC Users Specify Job Memory Requirements via Machine Learning
Adaptive Hybrid Queue Configuration for Supercomputer Systems
HPC System Lifetime Story: Workload Characterization and Evolutionary Analyses on NERSC Systems
SG-PBFS: Shortest Gap-Priority Based Fair Scheduling technique for job scheduling in cloud environment
Enterprise Resource Management in Mesos Clusters
Energy-aware workflow scheduling and optimization in clouds using bat algorithm
A novel cloud workflow scheduling algorithm based on stable matching game theory
Server-side log data analytics for I/O workload characterization and coordination on large shared storage systems
INSPECTOR: Data Provenance Using Intel Processor Trace (PT)
Adaptive scheduling of parallel jobs in spark streaming
p
Coflow scheduling in input-queued switches: Optimal delay scaling and algorithms
Number retrieved: 73
Number of relevant: 13
Number of relevant retrieved: 13
sensitivty: 100.0%
precision: 17.8%


In [ ]:
# save to excel

# filter the quasi gold standard
filtered_retreived_papers = [paper for paper in retrieved_papers if paper[5] not in quasi_gold_standard_doi]
df = pd.DataFrame(filtered_retreived_papers)

df.to_excel("retreived_papers.xlsx")

: 